# Basic stl2fem workflow

This notebook is intentionally small. It uses one tiny Nikolaisen2022 particle to demonstrate the package mechanics without filling the notebook with huge embedded 3D outputs. The full dataset notebooks are split into size bins separately.

**Unit caveat:** STL files do not reliably store physical units. They contain triangle coordinates as plain numbers. Here we treat the Nikolaisen2022 coordinates as micrometers by dataset convention, then write a Merrill-ready mesh with coordinates scaled to meters.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd()
if not (REPO / "src").exists() and (REPO.parent / "src").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from stl2fem.conversion import tetrahedralize_stl_for_merrill
from stl2fem.datasets import assign_size_bins, nikolaisen_inventory
from stl2fem.memory import estimate_merrill_memory
from stl2fem.quality import inspect_surface_stl, inspect_volume_mesh, load_surface, load_volume_mesh
from stl2fem.units import DEFAULT_TARGET_EDGE_LENGTH_M, add_meter_scaled_columns, make_unit_context

DATASET_ROOT = REPO / "data" / "Nikolaisen2022"
OUTPUT_ROOT = REPO / "processed" / "demo"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAMPLE_STL = DATASET_ROOT / "Plag Binary meshes" / "PLAG246-binary.stl"
INPUT_UNIT = "um"
TARGET_EDGE_LENGTH_M = DEFAULT_TARGET_EDGE_LENGTH_M

## Inventory and size bins

The dataset folders named `Binary meshes` are used as requested, but the package checks the actual STL encoding. Some files in those folders are ASCII STL text, so downstream logic should trust detection rather than folder names.

In [ ]:
inventory = assign_size_bins(nikolaisen_inventory(DATASET_ROOT), n_bins=4)
print(f"Meshes found: {len(inventory)}")
print(inventory.groupby(["phase", "stl_format"]).size())

summary = inventory.groupby("size_bin")["stl_size_mib"].agg(["count", "min", "median", "max"])
summary

In [ ]:
ax = inventory.plot.scatter(x="size_rank", y="stl_size_mib", c="size_bin_index", colormap="viridis", figsize=(7, 4))
ax.set_title("Nikolaisen2022 STL files sorted by file size")
ax.set_xlabel("Size rank")
ax.set_ylabel("STL size [MiB]")
plt.show()

## Unit context

The default physical target edge length is 9 nm, written as `9e-9 m`. Because this STL is interpreted as micrometers, the native Gmsh target is `0.009` coordinate units.

In [ ]:
units = make_unit_context(input_unit=INPUT_UNIT, target_edge_length_m=TARGET_EDGE_LENGTH_M)
pd.Series({
    "input_unit": units.input_unit,
    "input_scale_to_meters": units.input_scale_to_meters,
    "target_edge_length_m": units.target_edge_length_m,
    "target_edge_length_native": units.target_edge_length_native,
})

## Surface quality

Before meshing, inspect whether the surface is watertight, manifold, and single-component. The individual Nikolaisen particles are documented as repaired, but this check is still useful because FEM tetrahedralization assumes a closed surface.

In [ ]:
surface_quality = inspect_surface_stl(SAMPLE_STL)
pd.Series(surface_quality)[[
    "path", "n_points_surface", "n_triangles_surface", "surface_area", "surface_volume",
    "boundary_edge_count", "non_manifold_edge_count", "connected_components", "is_watertight",
]]

In [ ]:
import pyvista as pv

surface = load_surface(SAMPLE_STL)
plotter = pv.Plotter(notebook=True, window_size=(650, 450))
plotter.add_mesh(surface, color="lightsteelblue", show_edges=True)
plotter.add_axes()
plotter.show_grid()
plotter.show()

## Convert STL to a tetrahedral volume mesh

This call uses Gmsh's 3D Delaunay algorithm. It writes two meshes: a native-unit diagnostic mesh and a Merrill-ready mesh whose coordinates are scaled to meters. The `edge_length_*` columns below are realized tetrahedron edge lengths in native STL units; the `*_m` columns are the same lengths in meters.

In [ ]:
native_msh = OUTPUT_ROOT / "PLAG246_native.msh"
merrill_msh = OUTPUT_ROOT / "PLAG246_meters.msh"

tetrahedralize_stl_for_merrill(
    SAMPLE_STL,
    native_msh,
    merrill_msh,
    input_unit=INPUT_UNIT,
    target_edge_length_m=TARGET_EDGE_LENGTH_M,
    overwrite=True,
)
volume_quality = add_meter_scaled_columns(
    inspect_volume_mesh(native_msh),
    input_scale_to_meters=units.input_scale_to_meters,
)
memory = estimate_merrill_memory(volume_quality["n_nodes"], volume_quality["n_tets"])

report = {**volume_quality, **memory, "target_edge_length_m": TARGET_EDGE_LENGTH_M, "target_edge_length_native": units.target_edge_length_native}
pd.Series(report)[[
    "target_edge_length_m", "target_edge_length_native",
    "edge_length_median", "edge_length_p95", "edge_length_median_m", "edge_length_p95_m",
    "n_nodes", "n_tets", "msh_size_mib", "scaled_jacobian_min", "radius_ratio_max",
    "estimated_memory_human",
]]

In [ ]:
volume = load_volume_mesh(native_msh)
plotter = pv.Plotter(notebook=True, window_size=(650, 450))
plotter.add_mesh(volume, color="tomato", opacity=0.35, show_edges=True)
plotter.add_axes()
plotter.show_grid()
plotter.show()